# Demostración del agente planificador de horarios


### Introducción

El presente notebook tiene como propósito demostrar el funcionamiento del agente planificador de horarios desarrollado para el proyecto final de la asignatura Representación y Solución Automática de Problemas.

El proyecto aborda el problema de asignar bloques de clases a diferentes días, franjas horarias y salones, teniendo en cuenta las restricciones de docentes, grupos y espacios disponibles.

A través de este notebook se presenta la creación de una instancia del problema, la representación de sus estados, la aplicación de acciones y la ejecución de diferentes algoritmos de búsqueda.

Finalmente, se comparan los resultados obtenidos mediante métricas como el costo de la solución, el tiempo de ejecución, los nodos generados y los nodos expandidos.

## 1. Preparación del entorno

En esta sección se prepara el entorno de ejecución de Python y se establece la conexión con el proyecto.

Se utiliza la biblioteca pathlib para identificar la ubicación del proyecto y sys para agregar su directorio raíz a las rutas de búsqueda de módulos de Python.

Esto permite importar y utilizar las clases, funciones y algoritmos implementados dentro del paquete planificador.

In [2]:
from pathlib import Path
import sys

raiz = Path.cwd()

if not (raiz / "planificador").exists():
    raiz = raiz.parent

sys.path.insert(0, str(raiz))

import planificador

print("Proyecto conectado correctamente")
print("Ruta:", raiz)

Proyecto conectado correctamente
Ruta: c:\Users\Santiago\Downloads\Proyecto-Final-Representaci-n--main\Proyecto-Final-Representaci-n--main


## 2. Generación de una instancia sintética

Para realizar las demostraciones se crea una instancia sintética utilizando la función `generar_instancia()`, implementada en el módulo `planificador.io_datos.sintetico`.

La función recibe dos parámetros:

- El número de bloques de clase que tendrá la instancia.
- Una semilla que permite reproducir la misma generación aleatoria.

Para este ejercicio se utilizan 20 bloques y la semilla 0. También se obtiene una solución de referencia generada junto con la instancia.

In [3]:
from planificador.io_datos.sintetico import generar_instancia

instancia, solucion_referencia = generar_instancia(20, 0)

print("Instancia creada correctamente")
print("Número de bloques:", len(instancia.bloque))
print("Número de docentes:", len(instancia.docente))
print("Número de grupos:", len(instancia.grupo))
print("Número de salones:", len(instancia.salon))

Instancia creada correctamente
Número de bloques: 20
Número de docentes: 4
Número de grupos: 2
Número de salones: 3


### Características de la instancia generada

La instancia contiene los siguientes elementos:

- 20 bloques de clase.
- 4 docentes.
- 2 grupos.
- 3 salones.

Estos elementos constituyen los recursos y las actividades que el agente debe organizar para construir un horario.

La utilización de una semilla fija permite reproducir las mismas condiciones en diferentes ejecuciones y facilita la comparación de resultados.

## 3. Representación del entorno: clase Instancia

La clase Instancia representa la información general del problema de planificación de horarios.

Esta clase almacena los elementos que componen el entorno, incluyendo el *calendario*, los *bloques de clase*, los *docentes*, los *grupos*, los *salones* y los pesos utilizados para calcular los costos.

Sus atributos principales se pueden organizar en las siguientes categorías:

- `Calendario`: días y franjas horarias.
- `Recursos`: salones, docentes y grupos.
- `Actividades`: bloques de clase.
- `Configuración`: nombre de la instancia y pesos.
- `Índices de consulta`: diccionarios para acceder directamente a bloques, docentes, grupos y salones.

A continuación se inspeccionan los atributos y métodos públicos disponibles en la instancia creada.

In [4]:
print("Clase:", type(instancia).__name__)

print("\nAtributos:")
for atributo, valor in vars(instancia).items():
    print(f"- {atributo}: {type(valor).__name__}")

print("\nMétodos públicos:")
for nombre in dir(instancia):
    if not nombre.startswith("_"):
        elemento = getattr(instancia, nombre)
        if callable(elemento):
            print("-", nombre)

Clase: Instancia

Atributos:
- dias: tuple
- franjas: tuple
- salones: tuple
- docentes: tuple
- grupos: tuple
- bloques: tuple
- pesos: Pesos
- nombre: str
- bloque: dict
- docente: dict
- grupo: dict
- salon: dict

Métodos públicos:
- a_dict
- desde_dict
- estudiantes
- horas_bloque


### Métodos principales de Instancia

La clase cuenta con cuatro métodos públicos principales:

- `a_dict`: convierte la instancia en un diccionario que puede serializarse.
- `desde_dict()`: reconstruye una instancia a partir de un diccionario.
- `estudiantes()`: permite consultar el número de estudiantes asociados a un bloque.
- `horas_bloque()`: calcula la duración real en horas de un bloque según su franja de inicio.

Estos métodos permiten consultar, representar y recuperar la información del entorno.

In [5]:
import inspect

for nombre in ["a_dict", "desde_dict", "estudiantes", "horas_bloque"]:
    metodo = getattr(type(instancia), nombre)

    print(f"\nMétodo: {nombre}")
    print("Firma:", inspect.signature(metodo))
    print("Descripción:", inspect.getdoc(metodo))


Método: a_dict
Firma: (self) -> 'dict'
Descripción: Representación JSON-serializable (frozensets → listas ordenadas).

Método: desde_dict
Firma: (datos: 'dict') -> "'Instancia'"
Descripción: Inversa de ``a_dict``.

Método: estudiantes
Firma: (self, bloque_id: 'str') -> 'int'
Descripción: |E(m)|: estudiantes del bloque.

Método: horas_bloque
Firma: (self, bloque_id: 'str', franja_inicio: 'int') -> 'float'
Descripción: Horas reales de reloj de un bloque ubicado desde ``franja_inicio``.


### Consulta de información de un bloque

Se selecciona un bloque de la instancia para consultar algunas de sus características.

Mediante su identificador se accede directamente al diccionario de bloques y se obtiene información como su duración y el número de estudiantes asociados.

Esta consulta permite observar cómo se accede a los datos individuales que conforman el problema.

In [6]:
bloque_id = next(iter(instancia.bloque))
bloque = instancia.bloque[bloque_id]

print("ID del bloque:", bloque_id)
print("Duración:", bloque.duracion, "franjas")
print("Número de estudiantes:", instancia.estudiantes(bloque_id))

ID del bloque: B001
Duración: 1 franjas
Número de estudiantes: 11


### Cálculo de la duración real de un bloque

El método `horas_bloque()` permite obtener la duración de una clase en horas reales, utilizando su identificador y la franja horaria de inicio.

En este ejemplo se utiliza el bloque seleccionado anteriormente y se consulta su duración comenzando en la primera franja disponible.

In [7]:
franja_inicio = 0

if franja_inicio + bloque.duracion <= len(instancia.franjas):
    horas = instancia.horas_bloque(bloque_id, franja_inicio)
    print("Bloque:", bloque_id)
    print("Franja de inicio:", franja_inicio)
    print("Duración real:", horas, "horas")
else:
    print("No hay suficientes franjas para este bloque.")

Bloque: B001
Franja de inicio: 0
Duración real: 1.0 horas


### Serialización y reconstrucción de la instancia

La serialización permite convertir la instancia del problema en una estructura de datos que puede almacenarse o transferirse.

En este proyecto, el método `a_dict()` convierte la instancia en un diccionario, mientras que el método `desde_dict()` permite reconstruirla a partir de ese diccionario.

A continuación, se comprueba que la instancia reconstruida conserva la misma información que la original.

In [8]:
datos = instancia.a_dict()

instancia_recuperada = type(instancia).desde_dict(datos)

print("Bloques originales:", len(instancia.bloque))
print("Bloques recuperados:", len(instancia_recuperada.bloque))
print(
    "¿Se conservó toda la representación?",
    instancia_recuperada.a_dict() == datos
)

Bloques originales: 20
Bloques recuperados: 20
¿Se conservó toda la representación? True


### Interpretación

La reconstrucción conserva los 20 bloques de la instancia original y la comparación entre los diccionarios devuelve *True*.

Esto demuestra que la conversión y reconstrucción realizadas conservaron la representación de la instancia.

## 4. Representación de los estados del problema

La clase *Estado* representa una asignación parcial de los bloques de clase a posiciones del horario.

Cada estado contiene información sobre los bloques que ya fueron ubicados y aquellos que todavía están pendientes.

Se utilizan dos elementos principales:

- X: representa la asignación actual de bloques.
- U(X): representa el conjunto de bloques que todavía no han sido ubicados.

El estado también mantiene índices de ocupación que permiten consultar qué bloques utilizan determinados salones, franjas horarias, docentes y grupos.

Para comenzar la demostración se crea un estado vacío a partir de la instancia generada.

In [9]:
from planificador.core.estado import Estado

estado = Estado(instancia)

print("Estado inicial:", estado)
print("Bloques ubicados:", len(estado.asignacion))
print("Bloques sin ubicar:", len(estado.sin_ubicar()))
print("Posición del primer bloque:", estado.posicion(bloque_id))

Estado inicial: Estado('sint_20_0', ubicados=0, sin_ubicar=20)
Bloques ubicados: 0
Bloques sin ubicar: 20
Posición del primer bloque: None


### Interpretación del estado inicial

El resultado muestra que inicialmente no existen bloques asignados y que los 20 bloques permanecen sin ubicar.

La posición del bloque consultado devuelve *None*, debido a que todavía no ha sido asignado.

Este estado representa el punto de partida de los algoritmos de búsqueda.

## 5. Aplicación de acciones sobre el estado

Las acciones representan las operaciones que permiten modificar una asignación del horario.

Entre las acciones implementadas se encuentran:

- `Anadir`: incorpora un bloque a una posición.
- `Eliminar`: retira un bloque que ya está ubicado.
- `Reubicar`: cambia la posición de un bloque.
- `CambiarSalon`: modifica el salón asignado.
- `Intercambiar`: intercambia las posiciones de dos bloques.

La función `posiciones_legales()` genera las posiciones que cumplen las condiciones de legalidad relacionadas con la forma del horario, la compatibilidad del salón y su disponibilidad.

En esta demostración se selecciona una posición disponible y se utiliza la acción Anadir para asignar el bloque *B001*.

In [10]:
from planificador.core.modelo import Anadir
from planificador.core.acciones import (
    posiciones_legales,
    aplicar_en_sitio
)

estado = Estado(instancia)

bloque_id = "B001"

posiciones = posiciones_legales(estado, bloque_id)

print("Posiciones disponibles:", len(posiciones))

posicion = posiciones[0]
accion = Anadir(bloque_id, posicion)

accion_inversa = aplicar_en_sitio(estado, accion)

print("\nAcción aplicada:", accion)
print("Posición del bloque:", estado.posicion(bloque_id))
print("Bloques ubicados:", len(estado.asignacion))
print("Bloques sin ubicar:", len(estado.sin_ubicar()))

Posiciones disponibles: 96

Acción aplicada: Anadir(bloque='B001', posicion=Posicion(dia=0, franja=0, salon='S01'))
Posición del bloque: Posicion(dia=0, franja=0, salon='S01')
Bloques ubicados: 1
Bloques sin ubicar: 19


### Acción inversa y recuperación del estado

La función `aplicar_en_sitio()` modifica directamente el estado y devuelve una acción inversa que permite deshacer la operación realizada.

En este caso, después de añadir el bloque *B001*, se aplica su acción inversa para eliminarlo y recuperar el estado anterior.

Esta funcionalidad permite realizar modificaciones durante la búsqueda y regresar a estados previos sin reconstruir completamente la instancia.

In [11]:
aplicar_en_sitio(estado, accion_inversa)

print("Bloques ubicados:", len(estado.asignacion))
print("Bloques sin ubicar:", len(estado.sin_ubicar()))
print("Posición del bloque:", estado.posicion(bloque_id))

Bloques ubicados: 0
Bloques sin ubicar: 20
Posición del bloque: None


In [12]:
from planificador.algoritmos.registro import crear_solver
from planificador.algoritmos.base import Limites

## 6. Aplicación de algoritmos de búsqueda

### 6.1. Backtracking (búsqueda con retroceso)

**Backtracking** es una estrategia de búsqueda en profundidad que construye una solución mediante decisiones sucesivas.

En este proyecto se implementa como un problema de satisfacción de restricciones (CSP).

Su funcionamiento utiliza:

- `MRV`: selecciona primero la variable con el menor número de valores disponibles.
- `Grado`: permite priorizar variables que tienen mayor influencia sobre otras.
- `Forward Checking`: actualiza los dominios de las variables pendientes y detecta situaciones en las que una asignación impide continuar.
- `LCV`: organiza los valores para probar primero aquellos que restringen menos las opciones restantes.

En esta demostración se utiliza el modo *primera solución*, que finaliza cuando encuentra una asignación completa que satisface las restricciones duras.

In [13]:
solver = crear_solver("backtracking", modo="primera")

resultado = solver.resolver(
    instancia,
    limites=Limites(max_nodos=10000, max_segundos=5)
)

print("Algoritmo:", resultado.algoritmo)
print("¿Encontró solución?:", resultado.exito)
print("Costo total:", resultado.F)
print("Violaciones:", resultado.v)
print("Nodos expandidos:", resultado.nodos_expandidos)
print("Nodos generados:", resultado.nodos_generados)
print("Tiempo:", resultado.tiempo_s)
print("Terminación:", resultado.terminado_por)

Algoritmo: backtracking
¿Encontró solución?: True
Costo total: 25.0
Violaciones: 0
Nodos expandidos: 20
Nodos generados: 390
Tiempo: 0.011209000018425286
Terminación: meta


### Análisis de resultados de Backtracking

El algoritmo encontró una solución completa y válida para la instancia generada.

Se obtuvieron los siguientes resultados:

- Costo total F: 25.
- Violaciones: 0.
- Nodos expandidos: 20.
- Nodos generados: 390.
- Tiempo de ejecución: aproximadamente 0,0112 segundos.

El resultado demuestra que **Backtracking** pudo construir un horario sin violaciones duras para los 20 bloques.

Al utilizar el modo *primera solución*, el algoritmo no garantiza que el horario encontrado tenga el menor costo posible.

### 6.2. Búsqueda A*

A* es un algoritmo de búsqueda informada que utiliza el costo acumulado del camino y una estimación heurística del costo restante.

Su función de evaluación es:

f(n) = g(n) + h(n)

Donde:

- g(n): representa el costo acumulado desde el estado inicial.
- h(n): representa una estimación del costo necesario para completar el horario.

La implementación utiliza una heurística admisible y consistente. Con peso heurístico igual a 1, A* garantiza optimalidad respecto al costo aditivo g definido en el algoritmo.

Esta garantía no implica necesariamente encontrar el menor costo total F, debido a que algunos componentes del costo completo se calculan al finalizar el horario.

In [14]:
solver_astar = crear_solver("a_estrella")

resultado_astar = solver_astar.resolver(
    instancia,
    limites=Limites(max_nodos=100000, max_segundos=10)
)

print("Algoritmo:", resultado_astar.algoritmo)
print("¿Encontró solución?:", resultado_astar.exito)
print("Costo total:", resultado_astar.F)
print("Violaciones:", resultado_astar.v)
print("Nodos expandidos:", resultado_astar.nodos_expandidos)
print("Nodos generados:", resultado_astar.nodos_generados)
print("Tiempo:", round(resultado_astar.tiempo_s, 4))
print("Terminación:", resultado_astar.terminado_por)

Algoritmo: a_estrella
¿Encontró solución?: True
Costo total: 18.0
Violaciones: 0
Nodos expandidos: 20
Nodos generados: 394
Tiempo: 0.0207
Terminación: meta


### Análisis de resultados de A*

A* encontró una solución completa y válida.

Los resultados obtenidos fueron:

- Costo total F: 18.
- Violaciones: 0.
- Nodos expandidos: 20.
- Nodos generados: 394.
- Tiempo de ejecución: aproximadamente 0,0207 segundos.

En esta ejecución, A* obtuvo el menor costo total entre los algoritmos probados.

Sin embargo, este resultado corresponde a una sola instancia y no demuestra que el algoritmo haya encontrado el óptimo global respecto a F.

### 6.3. Búsqueda avara (Greedy)

**Greedy** es una estrategia de búsqueda informada que prioriza los estados que parecen estar más cerca de alcanzar el objetivo.

A diferencia de A*, no utiliza el costo acumulado del camino de la misma manera para determinar su prioridad.

Su selección depende principalmente de una función heurística que estima la distancia restante hasta completar el horario.

Esta estrategia puede encontrar soluciones rápidamente, pero no garantiza que la solución obtenida sea óptima.

In [15]:
solver_avara = crear_solver("avara")

resultado_avara = solver_avara.resolver(
    instancia,
    limites=Limites(max_nodos=100000, max_segundos=10)
)

print("Algoritmo:", resultado_avara.algoritmo)
print("¿Encontró solución?:", resultado_avara.exito)
print("Costo total:", resultado_avara.F)
print("Violaciones:", resultado_avara.v)
print("Nodos expandidos:", resultado_avara.nodos_expandidos)
print("Nodos generados:", resultado_avara.nodos_generados)
print("Tiempo:", round(resultado_avara.tiempo_s, 4))
print("Terminación:", resultado_avara.terminado_por)

Algoritmo: avara
¿Encontró solución?: True
Costo total: 36.0
Violaciones: 0
Nodos expandidos: 20
Nodos generados: 411
Tiempo: 0.021
Terminación: meta


### Análisis de resultados de Greedy

*Greedy* encontró una solución completa sin violaciones duras.

Los resultados fueron:

- Costo total F: 36.
- Violaciones: 0.
- Nodos expandidos: 20.
- Nodos generados: 411.
- Tiempo de ejecución: aproximadamente 0,021 segundos.

Aunque encontró una solución válida, su costo fue superior al obtenido por *Backtracking* y A*  en esta instancia.

Esto permite observar que encontrar una solución factible no significa necesariamente encontrar la solución de menor costo.

### 6.4. Búsqueda por haz (Beam Search)

**Beam Search** es una estrategia de búsqueda que explora el espacio de estados por niveles.

A diferencia de una búsqueda exhaustiva, mantiene únicamente una cantidad limitada de candidatos en cada nivel.

Esta cantidad se determina mediante el parámetro k, que en esta demostración tiene un valor de 5.

En cada nivel se evalúan los candidatos y se conservan los cinco mejores según la función de evaluación g(n) + h(n).

Esta limitación reduce la cantidad de nodos que se conservan en memoria, pero puede descartar caminos que conduzcan a soluciones de menor costo. Por esta razón, no garantiza optimalidad.

In [16]:
solver_beam = crear_solver("beam", ancho=5)

resultado_beam = solver_beam.resolver(
    instancia,
    limites=Limites(max_nodos=100000, max_segundos=10)
)

print("Algoritmo:", resultado_beam.algoritmo)
print("¿Encontró solución?:", resultado_beam.exito)
print("Costo total:", resultado_beam.F)
print("Violaciones:", resultado_beam.v)
print("Nodos expandidos:", resultado_beam.nodos_expandidos)
print("Nodos generados:", resultado_beam.nodos_generados)
print("Tiempo:", round(resultado_beam.tiempo_s, 4))
print("Terminación:", resultado_beam.terminado_por)

Algoritmo: beam
¿Encontró solución?: True
Costo total: 22.0
Violaciones: 0
Nodos expandidos: 96
Nodos generados: 1961
Tiempo: 0.0867
Terminación: meta


### Análisis de resultados de Beam Search

**Beam Search** encontró una solución completa y válida.

Los resultados obtenidos fueron:

- Ancho del haz: 5.
- Costo total F: 22.
- Violaciones: 0.
- Nodos expandidos: 96.
- Nodos generados: 1961.
- Tiempo de ejecución: aproximadamente 0,0867 segundos.

El algoritmo encontró una solución cuyo costo fue inferior al de **Greedy**, pero superior al de A* y **Backtracking** en esta ejecución.

Los resultados permiten observar el efecto de limitar la cantidad de candidatos explorados en cada nivel.

### 6.5. Búsqueda en profundidad sin heurísticas (DFS)

Para demostrar una estrategia de búsqueda no informada se utiliza **Backtracking** con sus mecanismos heurísticos desactivados.

La configuración utilizada es:

- MRV desactivado.
- Forward Checking desactivado.
- Orden natural de variables.
- Orden natural de valores.
- Sin posibilidad de saltar bloques.

De esta manera, la búsqueda sigue un recorrido en profundidad sin utilizar los mecanismos heurísticos de selección y poda que se emplean en la configuración habitual de **Backtracking**.

In [17]:
solver_dfs = crear_solver(
    "backtracking",
    modo="primera",
    mrv=False,
    forward_checking=False,
    permitir_saltar=False,
    orden_valores="natural"
)

resultado_dfs = solver_dfs.resolver(
    instancia,
    limites=Limites(max_nodos=10000, max_segundos=5)
)

print("Algoritmo: DFS sin heurísticas")
print("¿Encontró solución?:", resultado_dfs.exito)
print("Costo total:", resultado_dfs.F)
print("Violaciones:", resultado_dfs.v)
print("Nodos expandidos:", resultado_dfs.nodos_expandidos)
print("Nodos generados:", resultado_dfs.nodos_generados)
print("Tiempo:", round(resultado_dfs.tiempo_s, 4))
print("Terminación:", resultado_dfs.terminado_por)

Algoritmo: DFS sin heurísticas
¿Encontró solución?: False
Costo total: 5250.0
Violaciones: 8
Nodos expandidos: 10000
Nodos generados: 10319
Tiempo: 0.0417
Terminación: limite_nodos


### Análisis de resultados de DFS

En esta ejecución, **DFS** no consiguió encontrar una solución completa dentro del presupuesto establecido.

Los resultados fueron:

- Límite de nodos: 10.000.
- Nodos expandidos: 10.000.
- Nodos generados: 10.319.
- Violaciones: 8.
- Costo total F: 5250.
- Tiempo de ejecución: aproximadamente 0,0417 segundos.
- Motivo de terminación: límite de nodos.

El algoritmo se detuvo al alcanzar el presupuesto máximo de búsqueda.

Este resultado no significa que el problema carezca de solución, sino que esta estrategia no consiguió encontrarla dentro de los límites establecidos para la ejecución.

## 7. Comparación general de los algoritmos

A partir de los resultados obtenidos se puede realizar una comparación entre las diferentes estrategias de búsqueda utilizadas en la instancia de 20 bloques.

| Algoritmo | Solución válida | Costo F | Nodos expandidos |
|---|---|---:|---:|
| Backtracking | Sí | 25 | 20 |
| A* | Sí | 18 | 20 |
| Greedy | Sí | 36 | 20 |
| Beam Search | Sí | 22 | 96 |
| DFS sin heurísticas | No | 5250 | 10000 |

*Nota*: el costo F de **DFS** corresponde a una asignación incompleta, ya que el algoritmo terminó al alcanzar el límite de nodos. Por esta razón, no debe compararse directamente con los costos de las soluciones completas.

En esta ejecución, A* obtuvo el menor costo total entre las soluciones factibles encontradas. **Backtracking** resgistró el menor tiempo de ejecución en esta ejecución, mientras que **Beam Search** necesitó expandir más nodos para alcanzar su solución.

**Greedy** consiguió una solución válida, aunque con un costo superior al de los demás algoritmos que encontraron una solución factible.

Por otro lado, **DFS** alcanzó el límite de nodos sin completar el horario, lo que permite observar las dificultades que puede presentar una búsqueda en profundidad sin mecanismos heurísticos.

Estos resultados corresponden exclusivamente a una instancia sintética de 20 bloques y una semilla específica. Por lo tanto, no representan por sí solos el comportamiento general de los algoritmos.

Para obtener conclusiones más amplias se realizaron experimentos adicionales con diferentes tamaños, dificultades, semillas y escenarios, cuyos resultados se presentan en el análisis experimental del proyecto.

## 8. Conclusión

La demostración permitió observar cómo se representa el problema de planificación de horarios, cómo se modifican los estados mediante acciones y cómo diferentes algoritmos recorren el espacio de búsqueda.

La comparación evidencia que cada estrategia presenta ventajas y limitaciones relacionadas con el tiempo de ejecución, la cantidad de nodos explorados y la calidad de las soluciones obtenidas.